In [ ]:
!pip install -q transformers torch

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install -q emoji==0.6.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import os

MODEL_PATH = "/content/drive/MyDrive/resqai_bertweet_final"

print("Model folder exists:", os.path.exists(MODEL_PATH))
print("\nFiles in model folder:")

for root, dirs, files in os.walk(MODEL_PATH):
    level = root.replace(MODEL_PATH, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}  {file}")

Model folder exists: True

Files in model folder:
resqai_bertweet_final/
  config.json
  model.safetensors
  training_args.bin
  rng_state.pth
  trainer_state.json
  scaler.pt
  scheduler.pt
  optimizer.pt


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "vinai/bertweet-base",
    use_fast=False
)

print("BERTweet tokenizer loaded successfully!")

config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/843k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.91M [00:00<?, ?B/s]

BERTweet tokenizer loaded successfully!


In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

MODEL_PATH = "/content/drive/MyDrive/resqai_bertweet_final"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

print("Trained BERTweet model loaded successfully!")
print("Device:", device)
print("Number of classes:", model.config.num_labels)
print("Labels:", model.config.id2label)

Loading weights:   0%|          | 0/201 [00:04<?, ?it/s]

Trained BERTweet model loaded successfully!
Device: cpu
Number of classes: 10
Labels: {0: 'caution_and_advice', 1: 'displaced_people_and_evacuations', 2: 'infrastructure_and_utility_damage', 3: 'injured_or_dead_people', 4: 'missing_or_found_people', 5: 'not_humanitarian', 6: 'other_relevant_information', 7: 'requests_or_urgent_needs', 8: 'rescue_volunteering_or_donation_effort', 9: 'sympathy_and_support'}


In [ ]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 13.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.0 which is incompatible.


In [ ]:
from google import genai
import os

print("google-genai imported successfully!")

google-genai imported successfully!


In [ ]:
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key:")

Enter your Gemini API key:··········


In [ ]:
client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("Gemini client created successfully!")

Gemini client created successfully!


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class GeminiAnalysis(BaseModel):
    category: Literal[
        "caution_and_advice",
        "displaced_people_and_evacuations",
        "infrastructure_and_utility_damage",
        "injured_or_dead_people",
        "missing_or_found_people",
        "not_humanitarian",
        "other_relevant_information",
        "requests_or_urgent_needs",
        "rescue_volunteering_or_donation_effort",
        "sympathy_and_support"
    ]

    humanitarian: bool

    urgency: Literal[
        "LOW",
        "MEDIUM",
        "HIGH"
    ]

    request_for_help: bool

    location: str

    explanation: str

In [ ]:
from google.genai import types

def analyze_with_gemini(
    text,
    bertweet_prediction,
    bertweet_confidence
):

    prompt = f"""
You are the secondary analysis component of ResQAI,
an AI-assisted humanitarian disaster information system.

Analyze the following social-media message related to a possible
disaster or emergency.

MESSAGE:
{text}

The primary BERTweet classifier predicted:

Category: {bertweet_prediction}
Confidence: {bertweet_confidence:.2%}

Your task is to independently analyze the message and select the
MOST appropriate humanitarian category.

You MUST choose exactly one of the categories provided in the output schema.

Consider the actual meaning and context of the message rather than
relying only on disaster-related keywords.

Determine:
- the most appropriate humanitarian category
- whether the message is humanitarian/relevant
- urgency
- whether the author is requesting help
- any explicitly mentioned location
- a concise explanation for the classification

For LOCATION:
Return the explicitly mentioned location if one exists.
If no location is present, return "NONE".

Do not invent information that is not present in the message.
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=GeminiAnalysis
        )
    )

    return response.parsed

In [ ]:
test_message = """
Not sure if I slept thru aftershocks between 4-6am
or if theyre just starting up again in Wellington? #eqnz
"""

gemini_result = analyze_with_gemini(
    text=test_message,
    bertweet_prediction="other_relevant_information",
    bertweet_confidence=0.54
)

print(gemini_result)

category='other_relevant_information' humanitarian=True urgency='LOW' request_for_help=False location='Wellington' explanation='The post discusses potential earthquake aftershocks felt in Wellington, sharing personal observations about seismic activity.'


In [ ]:
def predict_bertweet(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    predicted_id = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    confidence = probabilities[0, predicted_id].item()

    predicted_label = model.config.id2label[predicted_id]

    return predicted_label, confidence

In [ ]:
def predict_resqai(text, confidence_threshold=0.70):

    # Get BERTweet prediction
    bertweet_label, confidence = predict_bertweet(text)

    # Send to Gemini if:
    # 1. BERTweet confidence is low, OR
    # 2. BERTweet predicts "not_humanitarian"
    send_to_gemini = (
        confidence < confidence_threshold
        or bertweet_label == "not_humanitarian"
    )

    # High-confidence BERTweet prediction
    if not send_to_gemini:

        return {
            "message": text,
            "model_used": "BERTweet",
            "final_category": bertweet_label,
            "bertweet_confidence": confidence,
            "gemini_used": False
        }

    # Low-confidence / potentially risky prediction
    gemini_result = analyze_with_gemini(
        text=text,
        bertweet_prediction=bertweet_label,
        bertweet_confidence=confidence
    )

    return {
        "message": text,
        "model_used": "BERTweet + Gemini",
        "bertweet_prediction": bertweet_label,
        "bertweet_confidence": confidence,
        "gemini_used": True,
        "final_category": gemini_result.category,
        "humanitarian": gemini_result.humanitarian,
        "urgency": gemini_result.urgency,
        "request_for_help": gemini_result.request_for_help,
        "location": gemini_result.location,
        "explanation": gemini_result.explanation
    }

In [ ]:
def display_resqai_result(result):

    print("=" * 60)
    print("RESQAI ANALYSIS")
    print("=" * 60)

    print("\nMESSAGE:")
    print(result["message"])

    print("\nMODEL:")
    print(result["model_used"])

    print("\nFINAL CATEGORY:")
    print(result["final_category"])

    print("\nBERTWEET CONFIDENCE:")
    print(f"{result['bertweet_confidence'] * 100:.2f}%")

    print("\nGEMINI USED:")
    print(result["gemini_used"])

    if result["gemini_used"]:
        print("\nHUMANITARIAN:")
        print(result["humanitarian"])

        print("\nURGENCY:")
        print(result["urgency"])

        print("\nREQUEST FOR HELP:")
        print(result["request_for_help"])

        print("\nLOCATION:")
        print(result["location"])

        print("\nEXPLANATION:")
        print(result["explanation"])

    print("=" * 60)

In [ ]:
test_message = """
Not sure if I slept thru aftershocks between 4-6am
or if theyre just starting up again in Wellington? #eqnz
"""

result = predict_resqai(test_message)

display_resqai_result(result)

RESQAI ANALYSIS

MESSAGE:

Not sure if I slept thru aftershocks between 4-6am
or if theyre just starting up again in Wellington? #eqnz


MODEL:
BERTweet + Gemini

FINAL CATEGORY:
other_relevant_information

BERTWEET CONFIDENCE:
73.90%

GEMINI USED:
True

HUMANITARIAN:
True

URGENCY:
LOW

REQUEST FOR HELP:
False

LOCATION:
Wellington

EXPLANATION:
The user is sharing observations and inquiring about aftershock activity in Wellington, which serves as general situational information.


In [ ]:
test_messages = [

    """
    We urgently need drinking water and medical supplies.
    There are hundreds of people trapped in the shelter.
    """,

    """
    People are going to get killed if they don't stay off roads.
    The driver drove into water under an overpass. #Harvey #Houston
    """,

    """
    My family is missing after the earthquake.
    We have not been able to contact them since yesterday.
    """,

    """
    The bridge has collapsed and several roads are completely destroyed.
    """,

    """
    Thank you to everyone who has donated and volunteered
    to help the victims of the earthquake.
    """
]

for i, message in enumerate(test_messages, 1):

    print(f"\n\n{'#' * 70}")
    print(f"TEST MESSAGE {i}")
    print(f"{'#' * 70}")

    result = predict_resqai(message)

    display_resqai_result(result)



######################################################################
TEST MESSAGE 1
######################################################################
RESQAI ANALYSIS

MESSAGE:

    We urgently need drinking water and medical supplies.
    There are hundreds of people trapped in the shelter.
    

MODEL:
BERTweet

FINAL CATEGORY:
requests_or_urgent_needs

BERTWEET CONFIDENCE:
94.20%

GEMINI USED:
False


######################################################################
TEST MESSAGE 2
######################################################################
RESQAI ANALYSIS

MESSAGE:

    People are going to get killed if they don't stay off roads.
    The driver drove into water under an overpass. #Harvey #Houston
    

MODEL:
BERTweet + Gemini

FINAL CATEGORY:
caution_and_advice

BERTWEET CONFIDENCE:
44.21%

GEMINI USED:
True

HUMANITARIAN:
True

URGENCY:
HIGH

REQUEST FOR HELP:
False

LOCATION:
Houston

EXPLANATION:
The tweet provides a warning and advice for people to stay 

In [ ]:
def analyze_tweet(text):

    result = predict_resqai(text)

    print("\n" + "=" * 70)
    print("                         RESQAI")
    print("           Humanitarian Disaster Message Analysis")
    print("=" * 70)

    print("\nMESSAGE")
    print("-" * 70)
    print(text.strip())

    print("\nPRIMARY CLASSIFIER")
    print("-" * 70)
    print("Model              :",
          "BERTweet" if not result["gemini_used"] else "BERTweet → Gemini")

    print("BERTweet Prediction:", result.get("bertweet_prediction",
                                           result["final_category"]))

    print("BERTweet Confidence:",
          f"{result['bertweet_confidence'] * 100:.2f}%")

    if result["gemini_used"]:

        print("\nSECONDARY ANALYSIS")
        print("-" * 70)

        print("Final Category     :", result["final_category"])
        print("Humanitarian       :", result["humanitarian"])
        print("Urgency            :", result["urgency"])
        print("Request for Help   :", result["request_for_help"])
        print("Location           :", result["location"])

        print("\nExplanation        :", result["explanation"])

    else:

        print("\nFINAL RESULT")
        print("-" * 70)
        print("Category           :", result["final_category"])
        print("Gemini Used        : No")

    print("\n" + "=" * 70)

    return result

In [ ]:
analyze_tweet(
    "People are trapped in the building and urgently need rescue."
)


                         RESQAI
           Humanitarian Disaster Message Analysis

MESSAGE
----------------------------------------------------------------------
People are trapped in the building and urgently need rescue.

PRIMARY CLASSIFIER
----------------------------------------------------------------------
Model              : BERTweet
BERTweet Prediction: requests_or_urgent_needs
BERTweet Confidence: 90.31%

FINAL RESULT
----------------------------------------------------------------------
Category           : requests_or_urgent_needs
Gemini Used        : No



{'message': 'People are trapped in the building and urgently need rescue.',
 'model_used': 'BERTweet',
 'final_category': 'requests_or_urgent_needs',
 'bertweet_confidence': 0.903063178062439,
 'gemini_used': False}

In [ ]:
[name for name in globals().keys() if "val" in name.lower()]

[]

In [ ]:
[x for x in globals().keys() if "dataset" in x.lower()]

[]